## Section 1: Setup & Imports

In [ ]:
import sys
from pathlib import Path
import shutil

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

# Force reload of modules
for module in list(sys.modules.keys()):
    if 'src' in module:
        del sys.modules[module]

# Clear old folders to regenerate
RAW_CHESTXRAY_OLD = project_root / 'data' / 'raw' / 'chestxray'

if RAW_CHESTXRAY_OLD.exists():
    shutil.rmtree(RAW_CHESTXRAY_OLD)
    print(f"Cleaned: {RAW_CHESTXRAY_OLD}")

print(f"Added to path: {project_root}")
print("Ready to process chest X-ray dataset")

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

print("Env set")

In [ ]:
# Dependencies
import os
from pathlib import Path
import json
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
import torchvision.transforms as T
import shutil, random, csv

print('Imports ready')

## Section 2: Download & Process Data

In [ ]:
# Chest X-ray Pneumonia Data Processing
print("=" * 70)
print("CHEST X-RAY PNEUMONIA DATA PROCESSING (2-class: Normal/Pneumonia)")
print("=" * 70)

# Setup paths
PROJECT_ROOT = Path.cwd().parent.parent
DOWNLOAD_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'chestxray_kaggle'
RAW_CHESTXRAY = PROJECT_ROOT / 'data' / 'raw' / 'chestxray'
SEED = 42

random.seed(SEED)
RAW_CHESTXRAY.mkdir(parents=True, exist_ok=True)

# Download dataset from Kaggle if not present
if not DOWNLOAD_ROOT.exists() or len(list(DOWNLOAD_ROOT.rglob('*.jpg'))) < 100:
    print(f"\nDownloading chest X-ray dataset from Kaggle...")
    try:
        import kagglehub
        download_path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
        print(f"Downloaded to: {download_path}")
        # Copy to our standard location if different
        if download_path != str(DOWNLOAD_ROOT):
            if DOWNLOAD_ROOT.exists():
                shutil.rmtree(DOWNLOAD_ROOT)
            shutil.copytree(download_path, DOWNLOAD_ROOT)
            print(f"Copied to: {DOWNLOAD_ROOT}")
    except Exception as e:
        print(f"Download failed: {e}")
        print(f"Please install kagglehub: pip install kagglehub")
        raise
else:
    print(f"✓ Using existing data: {DOWNLOAD_ROOT}")

# Check directory structure
print(f"\n✓ Dataset structure:")
for split in ['train', 'val', 'test']:
    split_path = DOWNLOAD_ROOT / split
    if split_path.exists():
        classes = [d.name for d in split_path.iterdir() if d.is_dir()]
        total = sum(len(list((split_path / c).glob('*'))) for c in classes if (split_path / c).is_dir())
        print(f"  {split}: {classes} ({total} total images)")

In [ ]:
# Convert to RAW_CHESTXRAY layout (2-class: NORMAL=0, PNEUMONIA=1)
print(f"\n✓ Converting to standard format...")
class_to_idx = {'NORMAL': 0, 'PNEUMONIA': 1}

for split in ['train', 'val', 'test']:
    split_src = DOWNLOAD_ROOT / split
    split_dst = RAW_CHESTXRAY / split
    
    if not split_src.exists():
        print(f"  ⚠ {split} not found in download, skipping")
        continue
    
    split_dst.mkdir(parents=True, exist_ok=True)
    labels_json = {}
    
    # Collect all images from class subdirectories
    for class_dir in split_src.iterdir():
        if not class_dir.is_dir():
            continue
        
        class_name = class_dir.name.upper()
        class_idx = class_to_idx.get(class_name, -1)
        
        if class_idx == -1:
            print(f"  ⚠ Unknown class: {class_name}")
            continue
        
        # Copy all images from this class
        for img_file in class_dir.glob('*'):
            if img_file.suffix.lower() not in ['.jpg', '.jpeg', '.png', '.gif']:
                continue
            
            dst_file = split_dst / img_file.name
            if not dst_file.exists():
                shutil.copyfile(img_file, dst_file)
            
            labels_json[img_file.name] = class_idx
    
    # Save labels.json
    with open(split_dst / 'labels.json', 'w') as f:
        json.dump(labels_json, f, indent=2)
    
    print(f"✓ {split}: {len(labels_json)} items")

print("\n" + "=" * 70)
print("✓ PROCESSING COMPLETE - Data ready for ChestXrayDataset!")
print("=" * 70)

## Section 3: Data Verification

In [ ]:
# Check data status and folder structure
print("=" * 60)
print("DATA STATUS CHECK")
print("=" * 60)

PROJECT_ROOT = Path.cwd().parent.parent
DOWNLOAD_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'chestxray_kaggle'
RAW_CHESTXRAY = PROJECT_ROOT / 'data' / 'raw' / 'chestxray'

print(f"\nProject root: {PROJECT_ROOT}")

# Check downloaded data
print(f"\n1. Downloaded data ({DOWNLOAD_ROOT.name}):")
if DOWNLOAD_ROOT.exists():
    images = list(DOWNLOAD_ROOT.rglob('*.jpg')) + list(DOWNLOAD_ROOT.rglob('*.jpeg'))
    print(f"   ✓ {len(images)} images found")
else:
    print(f"   ✗ Not found - download data to {DOWNLOAD_ROOT}")

# Check processed data
print(f"\n2. Processed data ({RAW_CHESTXRAY.name}):")
if RAW_CHESTXRAY.exists():
    for split in ['train', 'val', 'test']:
        split_dir = RAW_CHESTXRAY / split
        labels_file = split_dir / 'labels.json'
        if labels_file.exists():
            with open(labels_file) as f:
                labels = json.load(f)
                label_values = set(labels.values())
            print(f"   ✓ {split}: {len(labels)} items, labels {sorted(label_values)}")
else:
    print(f"   ✗ Not found - run cell above to process data")

print("\n" + "=" * 60)

## Section 4: Transforms & Dataset

In [ ]:
# Data transforms (ImageNet normalization for pretrained backbones)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

val_transform = T.Compose([
    T.Resize((256, 256)),
    T.CenterCrop(224),
    T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Class names
CLASS_NAMES = ['NORMAL', 'PNEUMONIA']
CLASS_IDX_TO_NAME = {i: name for i, name in enumerate(CLASS_NAMES)}

print('Transforms ready')
print(f'Classes: {CLASS_NAMES}')

In [ ]:
# Simple ChestXrayDataset
class ChestXrayDataset:
    def __init__(self, root_dir, split='train', transform=None):
        self.root = Path(root_dir) / split
        self.transform = transform
        self.samples = []
        
        # Load labels.json
        labels_file = self.root / 'labels.json'
        if labels_file.exists():
            with open(labels_file, 'r') as f:
                labels_dict = json.load(f)
            
            for img_name, label in labels_dict.items():
                img_path = self.root / img_name
                if img_path.exists():
                    self.samples.append((str(img_path), label))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return {'image': img, 'label': label}

# Test load
train_ds = ChestXrayDataset(str(RAW_CHESTXRAY), split='train', transform=val_transform)
print(f'Train samples: {len(train_ds)}')
print(f'Number of classes: {len(CLASS_NAMES)}')
print(f'Classes: {CLASS_NAMES}')

In [ ]:
# Get a batch with mixed classes
dl = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0)
batch = next(iter(dl))
    
if batch is not None:
    imgs = batch['image']
    labels = batch['label']
    print('Batch images shape:', imgs.shape)
    print('Batch labels:', labels.tolist())
    print('Batch class names:', [CLASS_IDX_TO_NAME[l] for l in labels.tolist()])
else:
    print('No samples found in train split')

## Section 5: Visualization

In [ ]:
# Visualize a few samples from different classes
if 'batch' in globals() and batch is not None:
    imgs = batch['image']
    labels = batch['label']
    inv_norm = T.Normalize(mean=[-m/s for m,s in zip(IMAGENET_MEAN, IMAGENET_STD)], std=[1/s for s in IMAGENET_STD])
    fig, axs = plt.subplots(2, 4, figsize=(14, 6))
    for i, ax in enumerate(axs.flatten()):
        if i >= imgs.size(0):
            break
        img = inv_norm(imgs[i]).permute(1, 2, 0).numpy()
        img = np.clip(img, 0, 1)
        ax.imshow(img, cmap='gray')
        label = labels[i].item()
        class_name = CLASS_IDX_TO_NAME.get(label, 'Unknown')
        ax.set_title(f'{class_name} (label={label})', fontsize=10, fontweight='bold')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No batch to visualize')